In [2]:
%cd '/home/mojtaba.kananisarche/Zh'

/home/mojtaba.kananisarche/Zh


In [3]:
from typing import Optional, List, Dict
import pandas as pd
from src.utils.file_folder import get_file_paths
import pandas as pd
import os

def _get_train_val_test_file_paths(
    input_path: str,
    primary_column_name: str,
    secondary_column_name: Optional[str] = None,
) -> List[Dict]:

    file_paths = get_file_paths(folder_path=input_path, depth=2, EXT_LIST=[".csv"])
    data = pd.DataFrame()
    data["path"] = file_paths
    data = data[~data['path'].apply(lambda x:os.path.split(x)[1].startswith('video_info'))].reset_index(drop=True)
    data["fold"] = (
        data["path"]
        .apply(lambda x: os.path.basename(x).split("_")[0])
        .astype(int)
        .astype(str)
    )
    data["subject_id"] = (
        data["path"]
        .apply(lambda x: os.path.basename(x).split("_")[1])
        .astype(int)
        .astype(str)
    )
    data = data.sort_values(["fold", "subject_id"]).reset_index(drop=True)

    columns_to_fold_by = data[
        [primary_column_name]
        + ([secondary_column_name] if secondary_column_name else [])
    ]

    train_val_test_file_paths = []
    unique_main_column_values = sorted(
        columns_to_fold_by[primary_column_name].unique()
    )

    if primary_column_name and secondary_column_name:
        for main_column_value in unique_main_column_values:
            corresponding_secondary_value = columns_to_fold_by[
                columns_to_fold_by[primary_column_name] == main_column_value
            ][secondary_column_name].unique()
            assert (
                len(corresponding_secondary_value) == 1
            ), f"Multiple Unique Secondary Values Found For Main Column Value {main_column_value} - {corresponding_secondary_value}"
            corresponding_secondary_value = corresponding_secondary_value[0]

            fold_train_val_test_file_paths = {"info": {}, "paths": {}}

            test_split = columns_to_fold_by[
                columns_to_fold_by[primary_column_name] == main_column_value
            ]
            fold_train_val_test_file_paths["paths"]["test"] = data.loc[
                test_split.index.to_list()
            ]["path"].to_list()
            fold_train_val_test_file_paths["info"]["test"] = {
                f"{primary_column_name}": sorted(
                    test_split[primary_column_name].unique().tolist()
                )
            }

            val_split = columns_to_fold_by[
                (columns_to_fold_by[primary_column_name] != main_column_value)
                & (
                    columns_to_fold_by[secondary_column_name]
                    == corresponding_secondary_value
                )
            ]
            fold_train_val_test_file_paths["paths"]["val"] = data.loc[
                val_split.index.to_list()
            ]["path"].to_list()
            fold_train_val_test_file_paths["info"]["val"] = {
                f"{primary_column_name}": sorted(
                    val_split[primary_column_name].unique().tolist()
                )
            }

            train_split = columns_to_fold_by[
                columns_to_fold_by[secondary_column_name]
                != corresponding_secondary_value
            ]
            fold_train_val_test_file_paths["paths"]["train"] = data.loc[
                train_split.index.to_list()
            ]["path"].to_list()
            fold_train_val_test_file_paths["info"]["train"] = {
                f"{primary_column_name}": sorted(
                    train_split[primary_column_name].unique().tolist()
                )
            }
            train_val_test_file_paths.append(fold_train_val_test_file_paths)

    elif primary_column_name:
        for main_column_value_i in unique_main_column_values:
            for main_column_value_j in unique_main_column_values:
                if main_column_value_i != main_column_value_j:
                    fold_train_val_test_file_paths = {"info": {}, "paths": {}}

                    test_split = columns_to_fold_by[
                        columns_to_fold_by[primary_column_name]
                        == main_column_value_i
                    ]
                    fold_train_val_test_file_paths["paths"]["test"] = data.loc[
                        test_split.index.to_list()
                    ]["path"].to_list()
                    fold_train_val_test_file_paths["info"]["test"] = {
                        f"{primary_column_name}": sorted(
                            test_split[primary_column_name].unique().tolist()
                        )
                    }

                    val_split = columns_to_fold_by[
                        columns_to_fold_by[primary_column_name]
                        == main_column_value_j
                    ]
                    fold_train_val_test_file_paths["paths"]["val"] = data.loc[
                        val_split.index.to_list()
                    ]["path"].to_list()
                    fold_train_val_test_file_paths["info"]["val"] = {
                        f"{primary_column_name}": sorted(
                            val_split[primary_column_name].unique().tolist()
                        )
                    }

                    train_split = columns_to_fold_by[
                        ~columns_to_fold_by[primary_column_name].isin(
                            [main_column_value_i, main_column_value_j]
                        )
                    ]
                    fold_train_val_test_file_paths["paths"]["train"] = data.loc[
                        train_split.index.to_list()
                    ]["path"].to_list()
                    fold_train_val_test_file_paths["info"]["train"] = {
                        f"{primary_column_name}": sorted(
                            train_split[primary_column_name].unique().tolist()
                        )
                    }

                    train_val_test_file_paths.append(fold_train_val_test_file_paths)

    return train_val_test_file_paths

In [4]:
input_directory_path = "/work/long_lab/Mojtaba/data/UTA/1.3.2-augmented_embedded_preprocessed"
primary_column_name = 'fold'
secondary_column_name = None
train_val_test_file_paths = _get_train_val_test_file_paths(
    input_path = input_directory_path, 
    primary_column_name = primary_column_name, 
    secondary_column_name = secondary_column_name, 
)

In [5]:
train_val_test_file_paths[0]['paths']['train']

['/work/long_lab/Mojtaba/data/UTA/1.3.2-augmented_embedded_preprocessed/brightness_alpha0.8_beta-10/3_25_5_30fps_brightness_alpha0.8_beta-10_preprocessed.csv',
 '/work/long_lab/Mojtaba/data/UTA/1.3.2-augmented_embedded_preprocessed/brightness_alpha0.8_beta-10/3_25_10_30fps_brightness_alpha0.8_beta-10_preprocessed.csv',
 '/work/long_lab/Mojtaba/data/UTA/1.3.2-augmented_embedded_preprocessed/brightness_alpha0.8_beta-10/3_25_0_30fps_brightness_alpha0.8_beta-10_preprocessed.csv',
 '/work/long_lab/Mojtaba/data/UTA/1.3.2-augmented_embedded_preprocessed/contrast_alpha0.7_beta0/3_25_5_30fps_contrast_alpha0.7_beta0_preprocessed.csv',
 '/work/long_lab/Mojtaba/data/UTA/1.3.2-augmented_embedded_preprocessed/contrast_alpha0.7_beta0/3_25_10_30fps_contrast_alpha0.7_beta0_preprocessed.csv',
 '/work/long_lab/Mojtaba/data/UTA/1.3.2-augmented_embedded_preprocessed/contrast_alpha0.7_beta0/3_25_0_30fps_contrast_alpha0.7_beta0_preprocessed.csv',
 '/work/long_lab/Mojtaba/data/UTA/1.3.2-augmented_embedded_pre

In [8]:
import random
from typing import List, Optional, Dict
from unittest import skip

import numpy as np
import pandas as pd
from keras.src.trainers.data_adapters.py_dataset_adapter import PyDataset
from src.preprocess.reshape import reshape_data_rnn
from src.utils.common import merge_intervals, pop_row
from src.utils.load_data import _extract_headers, load_data


class MyDataGenerator(PyDataset):
    """
    Custom data generator for Keras.
    """

    def __init__(self, 
                 paths                  : List[str],
                 video_info_csv_path    : str,
                 groupby_column         : str = "file_name",
                 sortby_column          : str = "frame",
                 target_column          : str = "target",
                 columns_to_drop        : List[str] = [],
                 batch_size             : int = 32,
                 slice_size             : int = 3000,
                 step_size              : int = 3000,
                 discard_threshold_ratio: float = 0.6,
                 shuffle                : bool = False,
                 **kwargs): 
        """
        Initialization function.  This is where you set up the basic parameters
        of your data generator.
        """
        super().__init__(**kwargs)

        self.paths                   = paths
        self.video_info_csv_path     = video_info_csv_path
        self.pointer                 = 0
        self.groupby_column          = groupby_column
        self.sortby_column           = sortby_column
        self.target_column           = target_column
        self.columns_to_drop         = columns_to_drop
        self.batch_size              = batch_size
        self.slice_size              = slice_size
        self.step_size               = step_size
        self.discard_threshold_ratio = discard_threshold_ratio
        self.shuffle                 = shuffle
        
        self.columns_to_drop_1       = self._get_columns_to_drop_1()
        self.feature_columns         = self._get_feature_columns()
        self.batch_info              = self._get_batch_info(
            video_info_csv_path     = self.video_info_csv_path,
            batch_size              = self.batch_size,
            slice_size              = self.slice_size,
            step_size               = self.step_size,
            groupby_column          = 'path',
            groupby_column_filter   = self.paths,
            counts_column           = 'total_frames',
            discard_threshold_ratio = self.discard_threshold_ratio)

        self.num_slices              = []

        self.on_epoch_end()

    def _get_columns_to_drop_1(self):
        return [
            column_name
            for column_name in self.columns_to_drop
            if column_name not in [self.groupby_column, 
                                   self.sortby_column, 
                                   self.target_column]
        ]
    
    def _get_feature_columns(self):
        return [
            column_name
            for column_name in _extract_headers(self.paths[0])
            if column_name not in 
            self.columns_to_drop + [self.target_column, 
                                    self.sortby_column, 
                                    self.groupby_column]
        ]

    def _get_batch_info(
        self,
        video_info_csv_path : str,
        batch_size : int,
        slice_size: int,
        step_size: Optional[int],
        groupby_column : str = 'path',
        groupby_column_filter : List[str] = [],
        counts_column : str = 'total_frames',
        discard_threshold_ratio: float = 0.6) -> Dict[str, List]:

        info = {
            'counts' : [],
            'batches': [],
        }

        video_info = load_data(video_info_csv_path)
        groupby_column_filter = groupby_column_filter or video_info[groupby_column].unique()
        video_info = video_info[video_info[groupby_column].isin(groupby_column_filter)]
        
        new_rows = []
        for _, row in video_info.iterrows():
            total = row[counts_column]
            start_frame = 0
            while start_frame < total:
                end_frame = min(start_frame + slice_size, total)
                current_slice_size = end_frame - start_frame
                if current_slice_size >= discard_threshold_ratio * slice_size:
                    new_row = row.copy()  # Create a copy to avoid modifying the original row
                    new_row['start_frame'] = start_frame
                    new_row['end_frame'] = end_frame
                    new_rows.append(new_row)
                start_frame += step_size

        video_info_processed = pd.DataFrame(new_rows).reset_index(drop=True)

        for group_identifier, group in video_info_processed.groupby(groupby_column):
            info['counts'].append({groupby_column : group_identifier, 
                                'count' : len(group)})

        for i in range(int(np.ceil(len(video_info_processed)/batch_size))):
            start_index = i * batch_size
            end_index   = min((i+1) * batch_size, len(video_info_processed))
            current_batch_info  = []
            for group_identifier, group in video_info_processed[start_index: end_index].groupby(groupby_column):
                current_batch_info.append(
                    {
                        groupby_column : group_identifier,
                        'raw_intervals' : list(zip(group['start_frame'], group['end_frame'])),
                        'merged_intervals' : merge_intervals(list(zip(group['start_frame'], group['end_frame'])))
                    }
                )
            info['batches'].append(current_batch_info)

        return info
    
    def __len__(self):
        """
        Returns the number of batches per epoch.  Keras uses this to know how many
        times to call `__getitem__` in one epoch.
        """
        assert len(self.batch_info['counts']) == len(self.paths), f"num_slices not calculated properly for all paths. num_slices : {len(self.batch_info['counts'])}, paths : {len(self.paths)}"

        return int(np.ceil(sum([file['count'] for file in self.batch_info['counts']])/self.batch_size))
    
    def _preprocess_data(self, file_data):
        file_data = file_data.drop(self.columns_to_drop_1, axis=1)
        X, y, g = reshape_data_rnn(df                      = file_data, 
                                   feature_columns         = self.feature_columns,
                                   slice_size              = self.slice_size,
                                   step_size               = self.step_size,
                                   groupby_column          = self.groupby_column,
                                   sortby_column           = self.sortby_column,
                                   target_column           = self.target_column,
                                   discard_threshold_ratio = self.discard_threshold_ratio)
        return X, y, g
    
    def on_epoch_end(self):
        """
        Called at the end of each epoch.
        #TODO implement proper shuffling. probably not needed as the batch indices are accessed without order
        """
        pass

    def __data_generation(self, index):
        """
        Generates data containing batch_size samples. This is where the actual data
        loading and preprocessing happens.
        """
        current_batch_info = self.batch_info['batches'][index]
        X, y, g = [], [], []

        for file in current_batch_info:
            file_data = load_data(input_path=file['path'], 
                      skiprows=file['merged_intervals'][0][0],
                      nrows=file['merged_intervals'][0][1] - file['merged_intervals'][0][0])
            X_file, y_file, g_file = self._preprocess_data(file_data=file_data)
            
            # Concatenate to the parent arrays
            X.extend(X_file)
            y.extend(y_file)
            g.extend(g_file)

        return {'X': np.array(X), 'y': np.array(y), 'g': np.array(g)}

    def __getitem__(self, index):
        """
        Generates one batch of data.  The core of the data generator.
        #TODO Calculate class weights based on the name of files and num_slices and return
        """
        print(f"index {index}")
        batch = self.__data_generation(index)
        return (batch['X'], batch['y'])

In [9]:
from keras.src.models.sequential import Sequential
import keras
from keras import layers

def _build_model(input_shape) -> Sequential:
    model = Sequential()
    model.add(layers.Input(shape=(input_shape[0], input_shape[1])))

    # Final output layer
    model.add(layers.Dense(1, activation="linear"))  # Use sigmoid activation

    model.compile(
        optimizer=keras.optimizers.AdamW(
            learning_rate=0.001,
            weight_decay=0.00001,
            # beta_1=beta_1,
            # beta_2=beta_2,
            # epsilon=epsilon
        ),
        loss="mse",
        metrics=["mae"],
    )
    return model

In [10]:
model = _build_model((1000, 128))

In [11]:
input_directory_path = "/work/long_lab/Mojtaba/data/UTA/1.3.2-augmented_embedded_preprocessed"
groupby_column, sortby_column, target_column = "file_name", "frame", "target"
video_info_csv_path = os.path.join(input_directory_path,'video_info_1.csv')
columns_to_drop = [
        "file_name",
        "file_name_1",
        "type",
        "frame",
        "fold",
        "subject_id",
    ]
batch_size              = 32
step_size               = 1000
slice_size              = 1000
discard_threshold_ratio = 0.5

train_data_gen = MyDataGenerator(
    paths                   = train_val_test_file_paths[0]['paths']['train'][:50],
    video_info_csv_path     = video_info_csv_path,
    groupby_column          = groupby_column,
    sortby_column           = sortby_column,
    target_column           = target_column,
    columns_to_drop         = columns_to_drop,
    batch_size              = batch_size,
    slice_size              = slice_size,
    step_size               = step_size,
    discard_threshold_ratio = discard_threshold_ratio,
    shuffle                 = True,
    workers                 = 1,
    use_multiprocessing     = False,
)
val_data_gen = MyDataGenerator(
    paths                   = train_val_test_file_paths[0]['paths']['val'][:50],
    video_info_csv_path     = video_info_csv_path,
    groupby_column          = groupby_column,
    sortby_column           = sortby_column,
    target_column           = target_column,
    columns_to_drop         = columns_to_drop,
    batch_size              = batch_size,
    slice_size              = slice_size,
    step_size               = step_size,
    discard_threshold_ratio = discard_threshold_ratio,
    shuffle                 = True,
    workers                 = 1,
    use_multiprocessing     = False,
)

2025-04-28 12:44:09,811 - INFO - Loaded file: /work/long_lab/Mojtaba/data/UTA/1.3.2-augmented_embedded_preprocessed/video_info_1.csv, data shape: (1575, 3)


2025-04-28 12:44:10,992 - INFO - Loaded file: /work/long_lab/Mojtaba/data/UTA/1.3.2-augmented_embedded_preprocessed/video_info_1.csv, data shape: (1575, 3)


In [12]:
model.fit(train_data_gen, validation_data=val_data_gen, verbose=1)

index 0


2025-04-28 12:44:14,768 - INFO - Loaded file: /work/long_lab/Mojtaba/data/UTA/1.3.2-augmented_embedded_preprocessed/brightness_alpha0.8_beta-10/3_25_5_30fps_brightness_alpha0.8_beta-10_preprocessed.csv, data shape: (18000, 135)
2025-04-28 12:44:20,733 - INFO - Loaded file: /work/long_lab/Mojtaba/data/UTA/1.3.2-augmented_embedded_preprocessed/brightness_alpha0.8_beta-10/3_26_0_30fps_brightness_alpha0.8_beta-10_preprocessed.csv, data shape: (14000, 135)


index 1


2025-04-28 12:44:22,025 - INFO - Loaded file: /work/long_lab/Mojtaba/data/UTA/1.3.2-augmented_embedded_preprocessed/brightness_alpha0.8_beta-10/3_25_10_30fps_brightness_alpha0.8_beta-10_preprocessed.csv, data shape: (18000, 135)
2025-04-28 12:44:25,080 - INFO - Loaded file: /work/long_lab/Mojtaba/data/UTA/1.3.2-augmented_embedded_preprocessed/brightness_alpha0.8_beta-10/3_26_0_30fps_brightness_alpha0.8_beta-10_preprocessed.csv, data shape: (4506, 135)
2025-04-28 12:44:25,519 - INFO - Loaded file: /work/long_lab/Mojtaba/data/UTA/1.3.2-augmented_embedded_preprocessed/brightness_alpha0.8_beta-10/3_26_10_30fps_brightness_alpha0.8_beta-10_preprocessed.csv, data shape: (9000, 135)


index 27


...

index 28


2025-04-28 12:49:50,268 - INFO - Loaded file: /work/long_lab/Mojtaba/data/UTA/1.3.2-augmented_embedded_preprocessed/contrast_alpha1.5_beta0/2_13_0_30fps_contrast_alpha1.5_beta0_preprocessed.csv, data shape: (18000, 135)
2025-04-28 12:49:51,505 - INFO - Loaded file: /work/long_lab/Mojtaba/data/UTA/1.3.2-augmented_embedded_preprocessed/contrast_alpha1.5_beta0/2_13_10_30fps_contrast_alpha1.5_beta0_preprocessed.csv, data shape: (10000, 135)
2025-04-28 12:49:52,189 - INFO - Loaded file: /work/long_lab/Mojtaba/data/UTA/1.3.2-augmented_embedded_preprocessed/cropped/2_14_10_30fps_cropped_preprocessed.csv, data shape: (3690, 135)


index 29


2025-04-28 12:49:52,774 - INFO - Loaded file: /work/long_lab/Mojtaba/data/UTA/1.3.2-augmented_embedded_preprocessed/contrast_alpha1.5_beta0/2_13_10_30fps_contrast_alpha1.5_beta0_preprocessed.csv, data shape: (8615, 135)
2025-04-28 12:49:53,816 - INFO - Loaded file: /work/long_lab/Mojtaba/data/UTA/1.3.2-augmented_embedded_preprocessed/contrast_alpha1.5_beta0/2_13_5_30fps_contrast_alpha1.5_beta0_preprocessed.csv, data shape: (20620, 135)


31/31 ━━━━━━━━━━━━━━━━━━━━ 329s 11s/step - loss: 42.7179 - mae: 5.8287 - val_loss: 34.3734 - val_mae: 5.1544


In [15]:
from src.utils.report import report

In [20]:
x = model.predict(val_data_gen)

index 0


2025-04-28 12:59:40,692 - INFO - Loaded file: /work/long_lab/Mojtaba/data/UTA/1.3.2-augmented_embedded_preprocessed/brightness_alpha0.8_beta-10/2_13_0_30fps_brightness_alpha0.8_beta-10_preprocessed.csv, data shape: (18000, 135)
2025-04-28 12:59:43,363 - INFO - Loaded file: /work/long_lab/Mojtaba/data/UTA/1.3.2-augmented_embedded_preprocessed/brightness_alpha0.8_beta-10/2_13_10_30fps_brightness_alpha0.8_beta-10_preprocessed.csv, data shape: (14000, 135)


index 1


2025-04-28 12:59:44,290 - INFO - Loaded file: /work/long_lab/Mojtaba/data/UTA/1.3.2-augmented_embedded_preprocessed/brightness_alpha0.8_beta-10/2_13_10_30fps_brightness_alpha0.8_beta-10_preprocessed.csv, data shape: (4615, 135)
2025-04-28 12:59:47,146 - INFO - Loaded file: /work/long_lab/Mojtaba/data/UTA/1.3.2-augmented_embedded_preprocessed/brightness_alpha0.8_beta-10/2_13_5_30fps_brightness_alpha0.8_beta-10_preprocessed.csv, data shape: (20620, 135)
2025-04-28 12:59:48,679 - INFO - Loaded file: /work/long_lab/Mojtaba/data/UTA/1.3.2-augmented_embedded_preprocessed/brightness_alpha0.8_beta-10/2_14_0_30fps_brightness_alpha0.8_beta-10_preprocessed.csv, data shape: (6000, 135)


index 0


...

index 28
28/30 ━━━━━━━━━━━━━━━━━━━━ 8s 4s/step 

2025-04-28 13:01:46,581 - INFO - Loaded file: /work/long_lab/Mojtaba/data/UTA/1.3.2-augmented_embedded_preprocessed/contrast_alpha1.5_beta0/2_13_0_30fps_contrast_alpha1.5_beta0_preprocessed.csv, data shape: (18000, 135)
2025-04-28 13:01:48,047 - INFO - Loaded file: /work/long_lab/Mojtaba/data/UTA/1.3.2-augmented_embedded_preprocessed/contrast_alpha1.5_beta0/2_13_10_30fps_contrast_alpha1.5_beta0_preprocessed.csv, data shape: (10000, 135)
2025-04-28 13:01:48,727 - INFO - Loaded file: /work/long_lab/Mojtaba/data/UTA/1.3.2-augmented_embedded_preprocessed/cropped/2_14_10_30fps_cropped_preprocessed.csv, data shape: (3690, 135)


index 29
29/30 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step

2025-04-28 13:01:49,271 - INFO - Loaded file: /work/long_lab/Mojtaba/data/UTA/1.3.2-augmented_embedded_preprocessed/contrast_alpha1.5_beta0/2_13_10_30fps_contrast_alpha1.5_beta0_preprocessed.csv, data shape: (8615, 135)
2025-04-28 13:01:50,243 - INFO - Loaded file: /work/long_lab/Mojtaba/data/UTA/1.3.2-augmented_embedded_preprocessed/contrast_alpha1.5_beta0/2_13_5_30fps_contrast_alpha1.5_beta0_preprocessed.csv, data shape: (20620, 135)


30/30 ━━━━━━━━━━━━━━━━━━━━ 122s 4s/step


In [23]:
x.flatten()

array([1.9571533 , 1.2991314 , 1.1405528 , ..., 0.02952561, 0.02952561,
       0.02952561], dtype=float32)

In [ ]:
y_val_pred, val_report = report(
    model=model,
    x=val_data_gen,
    y=val_data_gen.data["y"].to_list(),
    group_identifiers=val_data_gen.data["g"].to_list(),
    unique_classes=list(self.mapping.values()),
)

2025-04-28 08:18:20,004 - INFO - Loaded file: /work/long_lab/Mojtaba/data/UTA/1.3.2-augmented_embedded_preprocessed/video_info_1.csv, data shape: (1575, 3)


,file_name,path,total_frames
0,video_info_1.csv,/work/long_lab/Mojtaba/data/UTA/1.3.2-augmente...,18156
1,video_info_1.csv,/work/long_lab/Mojtaba/data/UTA/1.3.2-augmente...,18362
2,video_info_1.csv,/work/long_lab/Mojtaba/data/UTA/1.3.2-augmente...,18519
3,video_info_1.csv,/work/long_lab/Mojtaba/data/UTA/1.3.2-augmente...,19954
4,video_info_1.csv,/work/long_lab/Mojtaba/data/UTA/1.3.2-augmente...,16820
...,...,...,...
1570,video_info_1.csv,/work/long_lab/Mojtaba/data/UTA/1.3.2-augmente...,18206
1571,video_info_1.csv,/work/long_lab/Mojtaba/data/UTA/1.3.2-augmente...,17645
1572,video_info_1.csv,/work/long_lab/Mojtaba/data/UTA/1.3.2-augmente...,18081
1573,video_info_1.csv,/work/long_lab/Mojtaba/data/UTA/1.3.2-augmente...,20620


In [ ]:
from src.utils.common import merge_intervals
from typing import Optional, Dict
from src.utils.load_data import _extract_headers, load_data
import pandas as pd
import numpy as np

def get_batch_info(
    video_info_csv_path : str,
    batch_size : int,
    slice_size: int,
    step_size: Optional[int],
    groupby_column : str = 'path',
    counts_column : str = 'total_frames',
    discard_threshold_ratio: float = 0.6) -> Dict:

    info = {
        'counts' : [],
        'batch' : [],
    }

    video_info = load_data(video_info_csv_path)
    
    new_rows = []
    for _, row in video_info.iterrows():
        total = row[counts_column]
        start_frame = 0
        while start_frame < total:
            end_frame = min(start_frame + slice_size, total)
            current_slice_size = end_frame - start_frame
            if current_slice_size >= discard_threshold_ratio * slice_size:
                new_row = row.copy()  # Create a copy to avoid modifying the original row
                new_row['start_frame'] = start_frame
                new_row['end_frame'] = end_frame
                new_rows.append(new_row)
            start_frame += step_size

    video_info_processed = pd.DataFrame(new_rows).reset_index(drop=True)

    for group_identifier, group in video_info_processed.groupby(groupby_column):
        info['counts'].append({groupby_column : group_identifier, 
                               'count' : len(group)})

    for i in range(int(np.ceil(len(video_info_processed)/batch_size))):
        start_index = i * batch_size
        end_index   = min((i+1) * batch_size, len(video_info_processed))
        current_batch_info  = []
        for group_identifier, group in video_info_processed[start_index: end_index].groupby(groupby_column):
            current_batch_info.append(
                {
                    groupby_column : group_identifier,
                    'raw_intervals' : list(zip(group['start_frame'], group['end_frame'])),
                    'merged_intervals' : merge_intervals(list(zip(group['start_frame'], group['end_frame'])))
                }
            )
        info['batch'].append(current_batch_info)

    return info

In [70]:
import os 

input_directory_path = "/work/long_lab/Mojtaba/data/UTA/1.3.2-augmented_embedded_preprocessed"
video_info_csv_path = os.path.join(input_directory_path,'video_info_1.csv')

a, b = get_batch_info(
    video_info_csv_path = video_info_csv_path,
    batch_size = 32,
    slice_size= 1000,
    step_size= 1000,
    groupby_column = 'path',
    counts_column = 'total_frames',
    discard_threshold_ratio = 0.5)

2025-04-28 10:26:10,766 - INFO - Loaded file: /work/long_lab/Mojtaba/data/UTA/1.3.2-augmented_embedded_preprocessed/video_info_1.csv, data shape: (1575, 3)


In [71]:
b['batch']

[[{'path': '/work/long_lab/Mojtaba/data/UTA/1.3.2-augmented_embedded_preprocessed/brightness_alpha0.8_beta-10/1_08_0_30fps_brightness_alpha0.8_beta-10_preprocessed.csv',
   'raw_intervals': [(0, 1000),
    (1000, 2000),
    (2000, 3000),
    (3000, 4000),
    (4000, 5000),
    (5000, 6000),
    (6000, 7000),
    (7000, 8000),
    (8000, 9000),
    (9000, 10000),
    (10000, 11000),
    (11000, 12000),
    (12000, 13000),
    (13000, 14000),
    (14000, 15000),
    (15000, 16000),
    (16000, 17000),
    (17000, 18000)],
   'merged_intervals': [(0, 18000)]},
  {'path': '/work/long_lab/Mojtaba/data/UTA/1.3.2-augmented_embedded_preprocessed/brightness_alpha0.8_beta-10/1_09_5_30fps_brightness_alpha0.8_beta-10_preprocessed.csv',
   'raw_intervals': [(0, 1000),
    (1000, 2000),
    (2000, 3000),
    (3000, 4000),
    (4000, 5000),
    (5000, 6000),
    (6000, 7000),
    (7000, 8000),
    (8000, 9000),
    (9000, 10000),
    (10000, 11000),
    (11000, 12000),
    (12000, 13000),
    (13000,

In [40]:
a[::32]

,file_name,path,total_frames,start_frame,end_frame
0,video_info_1.csv,/work/long_lab/Mojtaba/data/UTA/1.3.2-augmente...,18156,0,1000
32,video_info_1.csv,/work/long_lab/Mojtaba/data/UTA/1.3.2-augmente...,18362,14000,15000
64,video_info_1.csv,/work/long_lab/Mojtaba/data/UTA/1.3.2-augmente...,19954,10000,11000


In [36]:
list(zip(i['start_frame'], i['end_frame']))

[(0, 1000),
 (1000, 2000),
 (2000, 3000),
 (3000, 4000),
 (4000, 5000),
 (5000, 6000),
 (6000, 7000),
 (7000, 8000),
 (8000, 9000),
 (9000, 10000),
 (10000, 11000),
 (11000, 12000),
 (12000, 13000),
 (13000, 14000)]

In [114]:
def merge_intervals(intervals):
    # First, sort the intervals by their start value
    intervals = sorted(intervals, key=lambda x: x[0])
    
    merged = []
    for interval in intervals:
        if not merged:
            merged.append(interval)
        else:
            last = merged[-1]
            if interval[0] <= last[1]:  # overlapping or touching
                merged[-1] = (last[0], max(last[1], interval[1]))  # merge
            else:
                merged.append(interval)
    return merged

# Example:
intervals = [(0, 10), (12, 15), (15, 25)]
print(merge_intervals(intervals))


[(0, 10), (12, 25)]


In [116]:
from src.utils.common import merge_intervals

ImportError: cannot import name 'merge_intervals' from 'src.utils.common' (/home/mojtaba.kananisarche/Zh/src/utils/common.py)

In [41]:
a.loc[0]

,file_name,path,total_frames,start_frame,end_frame
0,video_info_1.csv,/work/long_lab/Mojtaba/data/UTA/1.3.2-augmente...,18156,0,1000
0,video_info_1.csv,/work/long_lab/Mojtaba/data/UTA/1.3.2-augmente...,18156,1000,2000
0,video_info_1.csv,/work/long_lab/Mojtaba/data/UTA/1.3.2-augmente...,18156,2000,3000
0,video_info_1.csv,/work/long_lab/Mojtaba/data/UTA/1.3.2-augmente...,18156,3000,4000
0,video_info_1.csv,/work/long_lab/Mojtaba/data/UTA/1.3.2-augmente...,18156,4000,5000
0,video_info_1.csv,/work/long_lab/Mojtaba/data/UTA/1.3.2-augmente...,18156,5000,6000
0,video_info_1.csv,/work/long_lab/Mojtaba/data/UTA/1.3.2-augmente...,18156,6000,7000
0,video_info_1.csv,/work/long_lab/Mojtaba/data/UTA/1.3.2-augmente...,18156,7000,8000
0,video_info_1.csv,/work/long_lab/Mojtaba/data/UTA/1.3.2-augmente...,18156,8000,9000
0,video_info_1.csv,/work/long_lab/Mojtaba/data/UTA/1.3.2-augmente...,18156,9000,10000


In [154]:
dg.data

,X,y,g
0,,2.0,1_01_0_30fps_brightness_alpha0.8_beta-10.csv
1,,2.0,1_01_0_30fps_brightness_alpha0.8_beta-10.csv
2,,2.0,1_01_0_30fps_brightness_alpha0.8_beta-10.csv
3,,2.0,1_01_0_30fps_brightness_alpha0.8_beta-10.csv
4,,2.0,1_01_0_30fps_brightness_alpha0.8_beta-10.csv
5,,2.0,1_01_0_30fps_brightness_alpha0.8_beta-10.csv
6,,6.5,1_01_5_30fps_brightness_alpha0.8_beta-10.csv
7,,6.5,1_01_5_30fps_brightness_alpha0.8_beta-10.csv
8,,6.5,1_01_5_30fps_brightness_alpha0.8_beta-10.csv
9,,6.5,1_01_5_30fps_brightness_alpha0.8_beta-10.csv


In [147]:
dg.__getitem__(1)

2025-04-22 05:22:39,437 - INFO - Loaded file: /work/long_lab/Mojtaba/data/UTA/1.3.2-augmented_embedded_preprocessed/brightness_alpha0.8_beta-10/1_01_10_30fps_brightness_alpha0.8_beta-10_preprocessed.csv, data shape: (18027, 135)


,X,y,g
7,"[[6.5, 1.418, 0.618, 0.09, -0.28, 0.878, 1.212...",6.5,1_01_5_30fps_brightness_alpha0.8_beta-10.csv
8,"[[6.5, 1.693, 1.159, -0.187, -0.224, 0.834, 1....",6.5,1_01_5_30fps_brightness_alpha0.8_beta-10.csv
9,"[[6.5, 0.975, 0.407, 0.227, -0.552, 1.425, 0.6...",6.5,1_01_5_30fps_brightness_alpha0.8_beta-10.csv
10,"[[6.5, 1.078, 0.301, -0.296, 0.095, 1.409, 0.7...",6.5,1_01_5_30fps_brightness_alpha0.8_beta-10.csv
11,"[[6.5, 1.022, 0.204, -0.076, -0.152, 0.835, 0....",6.5,1_01_5_30fps_brightness_alpha0.8_beta-10.csv
12,"[[8.5, 0.723, 0.484, -0.139, 0.282, 1.075, 0.4...",8.5,1_01_10_30fps_brightness_alpha0.8_beta-10.csv
13,"[[8.5, 1.365, 0.639, -0.165, 0.032, 0.733, 0.5...",8.5,1_01_10_30fps_brightness_alpha0.8_beta-10.csv


In [148]:
dg.data

,X,y,g
0,"[[2.0, 0.804, 0.005, 0.232, 0.04, 0.599, 1.208...",2.0,1_01_0_30fps_brightness_alpha0.8_beta-10.csv
1,"[[2.0, 0.975, 0.315, 0.186, 0.484, 1.249, 0.43...",2.0,1_01_0_30fps_brightness_alpha0.8_beta-10.csv
2,"[[2.0, -0.137, 0.268, 0.413, 0.714, 1.186, -0....",2.0,1_01_0_30fps_brightness_alpha0.8_beta-10.csv
3,"[[2.0, 0.703, 0.149, 0.192, 0.053, 1.823, 0.55...",2.0,1_01_0_30fps_brightness_alpha0.8_beta-10.csv
4,"[[2.0, 0.237, 0.167, -0.129, 0.26, 1.162, 0.26...",2.0,1_01_0_30fps_brightness_alpha0.8_beta-10.csv
5,"[[2.0, 0.536, 0.584, 0.227, 0.304, 0.851, 0.31...",2.0,1_01_0_30fps_brightness_alpha0.8_beta-10.csv
6,"[[6.5, 1.412, 0.829, -0.153, -0.436, 1.186, 0....",6.5,1_01_5_30fps_brightness_alpha0.8_beta-10.csv
7,"[[6.5, 1.418, 0.618, 0.09, -0.28, 0.878, 1.212...",6.5,1_01_5_30fps_brightness_alpha0.8_beta-10.csv
8,"[[6.5, 1.693, 1.159, -0.187, -0.224, 0.834, 1....",6.5,1_01_5_30fps_brightness_alpha0.8_beta-10.csv
9,"[[6.5, 0.975, 0.407, 0.227, -0.552, 1.425, 0.6...",6.5,1_01_5_30fps_brightness_alpha0.8_beta-10.csv


In [149]:
dg.stash

,X,y,g
14,"[[8.5, -0.284, -0.589, 0.133, -0.255, 2.441, -...",8.5,1_01_10_30fps_brightness_alpha0.8_beta-10.csv
15,"[[8.5, 1.099, -0.034, 0.45, 0.533, 1.466, 0.14...",8.5,1_01_10_30fps_brightness_alpha0.8_beta-10.csv
16,"[[8.5, 1.115, 0.46, 0.475, -0.145, 1.436, 0.11...",8.5,1_01_10_30fps_brightness_alpha0.8_beta-10.csv
17,"[[8.5, 1.565, 0.08, 0.382, 0.291, 1.759, 0.405...",8.5,1_01_10_30fps_brightness_alpha0.8_beta-10.csv


In [33]:
for (X, y, g) in zip(a1, a2, a3):
    print(X, y, g)
    break

[[ 2.     0.804  0.005 ...  0.205 -1.507 -0.229]
 [ 2.     0.522 -0.369 ...  0.199 -1.104 -0.297]
 [ 2.     0.492 -0.019 ...  1.232 -0.433  0.135]
 ...
 [ 2.     0.619  0.678 ...  0.419 -1.322  0.369]
 [ 2.     0.562  0.629 ...  0.454 -1.341  0.35 ]
 [ 2.     0.792  0.627 ...  0.625 -1.295  0.273]] 2.0 1_01_0_30fps_brightness_alpha0.8_beta-10.csv


In [40]:
a2[0]

np.float64(2.0)

In [25]:
dg

NameError: name 'dg' is not defined

In [ ]:
keras.Sequential()
import inspect

print(inspect.getfile(Sequence))

/home/mojtaba.kananisarche/.conda/envs/.cvenv/lib/python3.9/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py


In [161]:
dg.stash.sample(len(dg.stash))

,X,y,g
8,"[[6.5, 1.693, 1.159, -0.187, -0.224, 0.834, 1....",6.5,1_01_5_30fps_brightness_alpha0.8_beta-10.csv
9,"[[6.5, 0.975, 0.407, 0.227, -0.552, 1.425, 0.6...",6.5,1_01_5_30fps_brightness_alpha0.8_beta-10.csv
10,"[[6.5, 1.078, 0.301, -0.296, 0.095, 1.409, 0.7...",6.5,1_01_5_30fps_brightness_alpha0.8_beta-10.csv
7,"[[6.5, 1.418, 0.618, 0.09, -0.28, 0.878, 1.212...",6.5,1_01_5_30fps_brightness_alpha0.8_beta-10.csv
11,"[[6.5, 1.022, 0.204, -0.076, -0.152, 0.835, 0....",6.5,1_01_5_30fps_brightness_alpha0.8_beta-10.csv


In [ ]:
data = pd.read_csv(os.path.join(input_path, 'video_info_1.csv'))

In [73]:
popped, remaining = pop_row(data, slice(0,5))

In [74]:
popped

,path,total_frames
0,/work/long_lab/Mojtaba/data/UTA/1.3.2-augmente...,18156
1,/work/long_lab/Mojtaba/data/UTA/1.3.2-augmente...,18362
2,/work/long_lab/Mojtaba/data/UTA/1.3.2-augmente...,18519
3,/work/long_lab/Mojtaba/data/UTA/1.3.2-augmente...,19954
4,/work/long_lab/Mojtaba/data/UTA/1.3.2-augmente...,16820


In [75]:
remaining

,path,total_frames
5,/work/long_lab/Mojtaba/data/UTA/1.3.2-augmente...,18222
6,/work/long_lab/Mojtaba/data/UTA/1.3.2-augmente...,15010
7,/work/long_lab/Mojtaba/data/UTA/1.3.2-augmente...,18481
8,/work/long_lab/Mojtaba/data/UTA/1.3.2-augmente...,18729
9,/work/long_lab/Mojtaba/data/UTA/1.3.2-augmente...,18471
...,...,...
1570,/work/long_lab/Mojtaba/data/UTA/1.3.2-augmente...,18206
1571,/work/long_lab/Mojtaba/data/UTA/1.3.2-augmente...,17645
1572,/work/long_lab/Mojtaba/data/UTA/1.3.2-augmente...,18081
1573,/work/long_lab/Mojtaba/data/UTA/1.3.2-augmente...,20620


In [90]:
# %cd '/home/mojtaba.kananisarche/'

# from Zh.src.utils.load_data import load_data
# import numpy as np
# import pandas as pd
# from typing import List, Optional, Tuple, Optional
# from Zh.src.utils.file_folder import create_folders
# import os
# import json
# import pandas as pd
# import dataframe_image as dfi
# from Zh.src.utils.visualize_results import confusion_matrix_results, table_results

In [91]:
# path = "/home/mojtaba.kananisarche/Zh/outputs/2025-04-20|09:08:09|RNN_FACENET_5CV"

In [92]:
# data = table_results(
# results_directory_path  = path,
# output_directory_path   = None,
# output_file_path_prefix = 'results',
# cmap                    = "Blues",
# reverse_cmap            = False,
# header_1_bg_color       = 'silver',
# header_1_text_color     = 'black',
# header_2_bg_color       = 'darkgrey',
# header_2_text_color     = 'black',
# maximum_tries           = 1,
# slice_or_video          = 'video', #slice, video
# video_voting_method     = 'class_average', #majority, class_average, average
# height                  = 2,
# width                   = 5,
# )

In [93]:
# data = confusion_matrix_results(
#     results_directory_path  = path,
#     unique_classes          = [2, 6.5, 8.5],
#     output_directory_path   = None,
#     output_file_name_prefix = "results",
#     slice_or_video          = 'slice',     #slice, video
#     video_voting_method     = 'average',   #majority, class_average, average
#     index                   = -1,          # positive for any index and negative for all
#     split                   = "all",       # train, val, test, all
#     cmap                    = "Greys",
#     maximum_tries           = 5,
#     corner_text_color       = "white",
#     corner_bg_color         = "IndianRed",
#     pred_true_text_color    = "black",
#     pred_true_bg_color      = "silver",
#     labels_bg_color         = "beige",
#     labels_text_color       = "black",
#     height                  = 5,
#     width                   = None,
# )